# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices (FAIR^2) Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset—containing survey and regression outputs for rangeland management in Northern Kenya—using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print a summary of metadata
meta = dataset.metadata
print(f"Dataset: {meta.name}\n\n{meta.description}")

## 2. Data Overview

Let's explore the available record sets (`cr:RecordSet`) and fields in this Croissant dataset.

Record sets, fields, and columns are identified by unique `@id` values. We'll list them to guide extraction and analysis:

In [ ]:
# List available record sets and their fields by their @id
record_sets = list(dataset.record_sets)  # mlcroissant exposes this property
if not record_sets:
    print("No record sets found in this dataset Croissant schema.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        print(f"  Name: {rs.name}")
        print(f"  Description: {rs.description}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id:36}  name: {field.name}")
        print()

# Show the IDs of all record sets for further usage
record_set_ids = [rs.id for rs in record_sets]
print("All record set @ids:", record_set_ids)

## 3. Data Extraction

Load all records from each record set into separate Pandas DataFrames.

Refer to the record set and field/column `@id`s from the overview above for correct referencing.

In [ ]:
# If record sets are present, load them as DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  {len(df)} records loaded.")
    print(f"  Fields (columns): {df.columns.tolist()}")
    print()

if dataframes:
    # Show the head of the first record set
    first_rs = record_set_ids[0]
    print(f"Preview of record set {first_rs}:")
    display(dataframes[first_rs].head())
else:
    print("No dataframes created (no record sets in package).")

## 4. Exploratory Data Analysis (EDA)

This section demonstrates filtering, normalization, and aggregation using fields and columns referenced by their `@id`.

We'll select a numeric field from the first available record set for demonstration. Please adapt the field IDs according to your dataset's actual field list (see section 2 for IDs and names).

If no record sets are present, this section will be skipped.

In [ ]:
import numpy as np

if dataframes:
    # Example: pick the first record set and try to select a numeric field (by @id)
    first_rs_id = record_set_ids[0]
    df = dataframes[first_rs_id]
    print(f"Active record set: {first_rs_id}")

    # Try to find a numeric field by simple type-inference
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # try any column with float/int styled values
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_fields.append(col)
            except:
                pass

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Selected numeric field for analysis: {numeric_field_id}")

        # Filtering: keep rows where field > threshold (arbitrarily chosen here as 10)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization: z-score
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping: try to find an object-type or categorical column to group by
        possible_group_fields = [col for col in df.columns if (df[col].dtype == 'object' and col != numeric_field_id)]
        if possible_group_fields:
            group_field_id = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found for EDA. Please inspect the dataframe and try other columns.")
else:
    print("No record sets available -- skipping EDA.")

## 5. Visualization

Let's plot the distribution of the numeric field used above, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of '{numeric_field_id}' in record set {first_rs_id}")
    plt.show()

    # If we created a grouped_df, plot group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated the use of the `mlcroissant` library to:
- Load and inspect a Croissant-based dataset via its schema URL.
- Review available record sets, fields, and their `@id`s for programmatic reference.
- Extract tabular data into pandas DataFrames for EDA.
- Perform simple data analysis tasks such as filtering, normalization, and grouping.
- Visualize field distributions and group-level summary statistics.

**Next steps:**
- Further analyze domain-specific variables or regression results.
- Use record set `@id` and field `@id` for reliable, maintainable pipelines.
- Connect with the full [FAIR^2 dataset documentation](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) for deeper insights.

For more advanced Croissant dataset workflows, see the [`mlcroissant` documentation](https://github.com/mlcommons/croissant).